In [86]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix
from sklearn.compose import ColumnTransformer

In [175]:
df = pd.read_csv(r'C:\Users\PL\Documents\ML data\invoice.csv')

In [176]:
df.describe()

,buisness_year,doc_id,document_create_date,document_create_date.1,due_in_date,posting_id,area_business,total_open_amount,baseline_create_date,invoice_id,isOpen
count,50000.000000,5.000000e+04,5.000000e+04,5.000000e+04,5.000000e+04,50000.0,0.0,50000.000000,5.000000e+04,4.999400e+04,50000.000000
mean,2019.305700,2.012238e+09,2.019351e+07,2.019354e+07,2.019368e+07,1.0,NaN,32337.021651,2.019354e+07,2.011340e+09,0.200000
std,0.460708,2.885235e+08,4.496041e+03,4.482134e+03,4.470614e+03,0.0,NaN,39205.975231,4.482701e+03,2.766335e+08,0.400004
min,2019.000000,1.928502e+09,2.018123e+07,2.018123e+07,2.018122e+07,1.0,NaN,0.720000,2.018121e+07,1.928502e+09,0.000000
25%,2019.000000,1.929342e+09,2.019050e+07,2.019051e+07,2.019052e+07,1.0,NaN,4928.312500,2.019050e+07,1.929342e+09,0.000000
50%,2019.000000,1.929964e+09,2.019091e+07,2.019091e+07,2.019093e+07,1.0,NaN,17609.010000,2.019091e+07,1.929964e+09,0.000000
75%,2020.000000,1.930619e+09,2.020013e+07,2.020013e+07,2.020022e+07,1.0,NaN,47133.635000,2.020013e+07,1.930619e+09,0.000000
max,2020.000000,9.500000e+09,2.020052e+07,2.020052e+07,2.020071e+07,1.0,NaN,668593.360000,2.020052e+07,2.960636e+09,1.000000


In [177]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   business_code           50000 non-null  object 
 1   cust_number             50000 non-null  object 
 2   name_customer           50000 non-null  object 
 3   clear_date              40000 non-null  object 
 4   buisness_year           50000 non-null  float64
 5   doc_id                  50000 non-null  float64
 6   posting_date            50000 non-null  object 
 7   document_create_date    50000 non-null  int64  
 8   document_create_date.1  50000 non-null  int64  
 9   due_in_date             50000 non-null  float64
 10  invoice_currency        50000 non-null  object 
 11  document type           50000 non-null  object 
 12  posting_id              50000 non-null  float64
 13  area_business           0 non-null      float64
 14  total_open_amount       50000 non-null

In [179]:
df['posting_date'] = pd.to_datetime(df['posting_date'])
df['due_in_date'] = pd.to_datetime(df['due_in_date'].astype('Int64').astype(str), errors='coerce')
df['arrears'] = (df['posting_date'] - df['due_in_date']).dt.days

In [180]:
df.head()

,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,invoice_currency,document type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen,arrears
0,U001,0200769623,WAL-MAR corp,2020-02-11 00:00:00,2020.0,1.930438e+09,2020-01-26,20200125,20200126,2020-02-10,USD,RV,1.0,NaN,54273.28,20200126.0,NAH4,1.930438e+09,0,-15
1,U001,0200980828,BEN E,2019-08-08 00:00:00,2019.0,1.929646e+09,2019-07-22,20190722,20190722,2019-08-11,USD,RV,1.0,NaN,79656.60,20190722.0,NAD1,1.929646e+09,0,-20
2,U001,0200792734,MDV/ trust,2019-12-30 00:00:00,2019.0,1.929874e+09,2019-09-14,20190914,20190914,2019-09-29,USD,RV,1.0,NaN,2253.86,20190914.0,NAA8,1.929874e+09,0,-15
3,CA02,0140105686,SYSC llc,NaN,2020.0,2.960623e+09,2020-03-30,20200330,20200330,2020-04-10,CAD,RV,1.0,NaN,3299.70,20200331.0,CA10,2.960623e+09,1,-11
4,U001,0200769623,WAL-MAR foundation,2019-11-25 00:00:00,2019.0,1.930148e+09,2019-11-13,20191113,20191113,2019-11-28,USD,RV,1.0,NaN,33133.29,20191113.0,NAH4,1.930148e+09,0,-15


In [181]:
X = df[[
    'business_code',
    'buisness_year',
    'invoice_currency',
    'document type',
    'total_open_amount',
    'cust_payment_terms',
    'arrears'
]]
y = df['isOpen']

In [182]:
X.head()

,business_code,buisness_year,invoice_currency,document type,total_open_amount,cust_payment_terms,arrears
0,U001,2020.0,USD,RV,54273.28,NAH4,-15
1,U001,2019.0,USD,RV,79656.60,NAD1,-20
2,U001,2019.0,USD,RV,2253.86,NAA8,-15
3,CA02,2020.0,CAD,RV,3299.70,CA10,-11
4,U001,2019.0,USD,RV,33133.29,NAH4,-15


In [183]:
X_train, X_test, y_train, y_test = train_test_split (
    X, y, test_size = 0.25, random_state = 42, stratify = y)

In [184]:
numeric_features = ['buisness_year','total_open_amount','arrears']
categorical_features = ['business_code','invoice_currency','document type','cust_payment_terms']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant',fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])
pipeline = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model', LogisticRegression(max_iter=5000, class_weight='balanced'))
])

In [185]:
param_grid = {'model__C': [0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='recall'
)

grid_search.fit(X_train, y_train)
grid_search.best_params_

{'model__C': 0.01}

In [186]:
pred = grid_search.predict(X_test)

In [187]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

In [188]:
print('accuracy',accuracy_score(y_test, pred))
print('precision',precision_score(y_test, pred))
print('recall',recall_score(y_test, pred))
print(confusion_matrix(y_test,pred))

accuracy 0.89584
precision 0.6575486586007364
recall 1.0
[[8698 1302]
 [   0 2500]]


In [189]:
probs = grid_search.predict_proba(X_test)[:,1]

In [301]:
pred = (probs >= 0.76).astype(int)

In [302]:
print(confusion_matrix(y_test,pred))

[[8699 1301]
 [   0 2500]]
